# 示例 01 · LLM 工具调用（ReAct 智能体）

**来源：** [LangGraph 教程 — 构建 ReAct 智能体](https://langchain-ai.github.io/langgraph/tutorials/introduction/)

本笔记本使用 LangGraph Graph API 构建一个 **ReAct（推理 + 行动）智能体**。
智能体通过逐步调用工具来解决多步骤算术问题，并对每步结果进行推理。

请从上到下依次运行每个单元格——每个单元格都依赖上一个的执行结果。

## 核心概念

### ReAct 循环
ReAct 智能体在"推理"（调用 LLM）和"行动"（执行工具）之间交替运行：

```
开始 → llm_call ──（有工具调用？）──► tool_node ──► llm_call
               └──（无工具调用）──► 结束
```

### LangGraph StateGraph（状态图）
智能体是一个包含两个节点的有向图：
- **`llm_call`** — 将完整消息历史发送给 LLM，追加其响应
- **`tool_node`** — 执行最新 AI 消息中的所有工具调用，追加结果

**状态累积机制：** `AgentState.messages` 使用 `operator.add` 作为归约器，
每个节点的输出会*追加*而非替换原有消息。LLM 始终能看到完整的对话历史。

In [ ]:
import sys
from pathlib import Path

# 将项目根目录加入 sys.path，使 common 包可被导入
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import operator
from typing import Literal
from typing_extensions import TypedDict, Annotated

from langchain_core.messages import AnyMessage, SystemMessage, ToolMessage, HumanMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END

from common.env import get_env  # noqa: F401 — 触发 .env 加载
from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

print("✓ 所有依赖导入完成")

## 第一步 · 定义工具

工具是用 `@tool` 装饰器修饰的普通 Python 函数。
LangChain 会读取函数的类型注解和文档字符串，自动生成 LLM 决策调用时所需的 JSON Schema。

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """将 a 乘以 b。"""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """将 a 与 b 相加。"""
    return a + b

@tool
def divide(a: int, b: float) -> float:
    """将 a 除以 b。"""
    return a / b

# 工具名称到工具对象的映射，供 tool_node 按名称分发调用
_tools = [add, multiply, divide]
_tools_by_name = {t.name: t for t in _tools}

print(f"✓ 已定义 {len(_tools)} 个工具：{[t.name for t in _tools]}")

## 第二步 · 构建智能体图

以下单元格将四个部分组装在一起：
1. **`AgentState`** — 保存完整消息历史的 TypedDict
2. **节点函数** — `llm_call` 和 `tool_node`
3. **路由函数** — `should_continue` 决定下一个节点
4. **`StateGraph`** — 将所有部分连接并编译为可运行对象

In [ ]:
# ── 状态定义 ───────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    # operator.add 表示每个节点的输出追加到列表，而非替换
    messages: Annotated[list[AnyMessage], operator.add]

# ── 绑定工具的模型（只初始化一次，避免重复创建）─────────────────────────
_model = create_llm().bind_tools(_tools)

_SYSTEM_PROMPT = (
    "You are a math assistant that uses tools for arithmetic. "
    "Call one tool at a time and wait for the result before deciding the next step."
)

# ── 节点函数 ───────────────────────────────────────────────────────────────
def llm_call(state: AgentState) -> dict:
    """将完整消息历史传给 LLM 并获取响应。"""
    response = _model.invoke([SystemMessage(content=_SYSTEM_PROMPT)] + state["messages"])
    return {"messages": [response]}

def tool_node(state: AgentState) -> dict:
    """执行最新 AI 消息中的所有工具调用。"""
    results = []
    for tc in state["messages"][-1].tool_calls:
        observation = _tools_by_name[tc["name"]].invoke(tc["args"])
        # 附带工具名称，便于 LLM 在并行调用时匹配结果
        results.append(ToolMessage(content=str(observation), tool_call_id=tc["id"], name=tc["name"]))
    return {"messages": results}

# ── 路由函数 ───────────────────────────────────────────────────────────────
def should_continue(state: AgentState) -> Literal["tool_node", "__end__"]:
    """若最新消息包含待执行的工具调用则继续，否则结束。"""
    return "tool_node" if state["messages"][-1].tool_calls else END

# ── 图构建 ─────────────────────────────────────────────────────────────────
def build_agent():
    g = StateGraph(AgentState)
    g.add_node("llm_call", llm_call)
    g.add_node("tool_node", tool_node)
    g.add_edge(START, "llm_call")
    g.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
    g.add_edge("tool_node", "llm_call")
    return g.compile()

print("✓ 智能体图定义完成，准备编译")

## 第三步 · 运行智能体

智能体将逐步推理：
1. LLM → 调用 `multiply(12345, 17)` → 结果 `209865`
2. LLM → 调用 `divide(209865, 3)` → 结果 `69955.0`
3. LLM → 无更多工具调用 → 返回最终答案

每个 `.pretty_print()` 展示对话链中的一条消息。

In [ ]:
agent = build_agent()
question = "What is 12345 multiplied by 17, then divided by 3?"

print(f"问题：{question}")
print("=" * 60)

handler = create_langfuse_handler()
config = build_langfuse_config(
    handler,
    session_id="s_01_notebook_cn",
    user_id="demo-user",
    trace_name="笔记本：数学计算",
)
config["configurable"] = {"thread_id": "nb-s01-cn"}

result = agent.invoke({"messages": [HumanMessage(content=question)]}, config=config)

for msg in result["messages"]:
    msg.pretty_print()

print(f"\n追踪记录：{get_langfuse_host()}")